# Ultimate Tic-Tac-Toe — Improved Evaluation ResNet (v3)

## Améliorations vs v2

### Bugs corrigés
- **Bug critique** : `exp_vals.sum` → `exp_vals.sum()` dans `compute_policy_target`
- **Encoding** : simplifié et cohérent (2 plans globaux joueur + métadonnées)

### Améliorations architecture
- **Réseau réduit** : 6 blocs × 128 filtres (~2M params vs 12M) → 6x plus rapide à l'inférence
- **Dropout 0.1** dans les têtes pour régulariser
- **SE blocks (Squeeze-Excitation)** optionnel pour améliorer la capacité sans coût majeur

### Améliorations données
- **Toutes les données** utilisées (~5.6M train au lieu de 1M)
- **Symétrie** : augmentation par les 8 symétries du plateau carré (rotations + réflexions)
 - Implémentée avec validation rigoureuse
- **Pré-processing offline** : cache des tenseurs pour éviter de recalculer à chaque epoch

### Améliorations entraînement
- **Warm-up** : 3 époques linéaires avant CosineAnnealing
- **Label smoothing** (ε=0.05) sur la policy target
- **Gradient clipping** conservé
- **Mixed precision** (fp16) pour accélérer l'entraînement sur GPU

### Métriques de suivi
- Loss + Pearson correlation + Directional accuracy à chaque epoch


In [1]:
"""import subprocess
result = subprocess.run(
    ["pip", "uninstall", "torch", "torchvision", "torchaudio", "-y"],
    capture_output=True, text=True
)
print(result.stdout)
print(result.stderr)

result = subprocess.run(
    ["pip", "install", "torch", "torchvision",
     "--index-url", "https://download.pytorch.org/whl/cu124"],
    capture_output=True, text=True
)
print(result.stdout[-3000:])  # les 3000 derniers chars pour éviter le flood
print(result.stderr[-1000:])"""

'import subprocess\nresult = subprocess.run(\n    ["pip", "uninstall", "torch", "torchvision", "torchaudio", "-y"],\n    capture_output=True, text=True\n)\nprint(result.stdout)\nprint(result.stderr)\n\nresult = subprocess.run(\n    ["pip", "install", "torch", "torchvision",\n     "--index-url", "https://download.pytorch.org/whl/cu124"],\n    capture_output=True, text=True\n)\nprint(result.stdout[-3000:])  # les 3000 derniers chars pour éviter le flood\nprint(result.stderr[-1000:])'

In [2]:
# ============================================================
# 1. IMPORTS & CONFIGURATION
# ============================================================
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR, CosineAnnealingLR, SequentialLR
from torch.cuda.amp import GradScaler, autocast
from datasets import load_dataset
from tqdm.auto import tqdm
import scipy.stats
import pytest

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

CFG = dict(
    # Réseau
    in_channels      = 8,    # 2 plans joueur global + meta_board + active_board + turn + 3 plans legaux
    num_filters      = 128,
    num_res_blocks   = 6,
    policy_size      = 81,
    use_se           = True,   # Squeeze-Excitation blocks
    dropout          = 0.1,
    # Entraînement
    batch_size       = 4096,
    lr               = 3e-4,
    weight_decay     = 1e-4,
    epochs           = 20,
    warmup_epochs    = 3,
    lambda_policy    = 1.0,
    label_smoothing  = 0.05,
    # Données
    min_visits       = 999_999,
    max_train_samples= 2_500_000,   # Utiliser 2,5M de données pour baisser le temps d'entraînement
    use_symmetry     = True,   # augmentation par symétries
    seed             = 42,
)
torch.manual_seed(CFG["seed"])
np.random.seed(CFG["seed"])
print(f"Config : {CFG}")

Device : cuda
GPU     : Tesla P100-PCIE-16GB
VRAM    : 17.1 GB
Config : {'in_channels': 8, 'num_filters': 128, 'num_res_blocks': 6, 'policy_size': 81, 'use_se': True, 'dropout': 0.1, 'batch_size': 4096, 'lr': 0.0003, 'weight_decay': 0.0001, 'epochs': 20, 'warmup_epochs': 3, 'lambda_policy': 1.0, 'label_smoothing': 0.05, 'min_visits': 999999, 'max_train_samples': 2500000, 'use_symmetry': True, 'seed': 42}


In [3]:
# ============================================================
# 2. ENCODING — SIMPLIFIE ET CORRECT
# ============================================================
# Format state string (93 chars):
#   s[0:81]  -> plateau principal  (0=vide, 1=joueur1, 2=joueur2)
#   s[81:90] -> meta-board (etat des 9 sous-grilles: 0=libre,1=P1,2=P2,3=nul)
#   s[90]    -> active board index (0-8, autre=toutes libres)
#   s[91]    -> current player (1 ou 2)
#   s[92]    -> profondeur (ignoree)

# ENCODING : 8 plans 9×9
# Plan 0    : cases P1 (global 9×9)
# Plan 1    : cases P2 (global 9×9)
# Plans 2-3 : meta-board P1, P2 (chaque grande case broadcastée sur sa sous-grille 3×3)
# Plan 4    : cases légales (1 si la case est jouable)
# Plan 5    : active_board (toutes les cases de la sous-grille active = 1, ou tout si libre)
# Plan 6    : turn (1.0 si P1, 0.0 si P2) — constant sur tout le plan
# Plan 7    : constante 1.0 (biais implicite)

def decode_state_string(s):
    """Parse la string 93-chars en composants."""
    n = len(s)
    board = np.array([int(c) for c in s[0:81]], dtype=np.int8)
    
    meta_board = np.zeros(9, dtype=np.int8)
    if n >= 90:
        meta_board = np.array([int(c) for c in s[81:90]], dtype=np.int8)
    
    active_idx = -1
    if n >= 91:
        raw = int(s[90])
        active_idx = raw if 0 <= raw <= 8 else -1
    
    current_player = 1
    if n >= 92:
        cp = int(s[91])
        current_player = cp if cp in (1, 2) else 1
    else:
        n_p1 = int((board == 1).sum())
        n_p2 = int((board == 2).sum())
        current_player = 1 if n_p1 == n_p2 else 2
    
    return board, meta_board, active_idx, current_player


def compute_legal_mask(board, meta_board, active_idx):
    """Calcule les cases légales comme masque 81-dim."""
    legal = np.zeros(81, dtype=np.float32)
    
    def sub_done(sub):
        return meta_board[sub] != 0
    
    def can_play_in(sub):
        return not sub_done(sub)
    
    if active_idx == -1:
        # Toutes les sous-grilles non terminées
        active_subs = [s for s in range(9) if can_play_in(s)]
    else:
        if can_play_in(active_idx):
            active_subs = [active_idx]
        else:
            active_subs = [s for s in range(9) if can_play_in(s)]
    
    for sub in active_subs:
        for cell in range(9):
            idx = sub * 9 + cell
            if board[idx] == 0:
                legal[idx] = 1.0
    
    return legal


def board_to_9x9(flat_board, player):
    """Convertit le plateau 81 en grille 9×9 pour un joueur."""
    grid = np.zeros((9, 9), dtype=np.float32)
    for sub in range(9):
        r_sub, c_sub = divmod(sub, 3)
        for cell in range(9):
            r_cell, c_cell = divmod(cell, 3)
            idx = sub * 9 + cell
            row = r_sub * 3 + r_cell
            col = c_sub * 3 + c_cell
            if flat_board[idx] == player:
                grid[row, col] = 1.0
    return grid


def meta_to_9x9(meta_board, player):
    """Broadcast le meta-board : chaque sous-grille 3×3 = 1 si gagnée par player."""
    grid = np.zeros((9, 9), dtype=np.float32)
    for sub in range(9):
        if meta_board[sub] == player:
            r_sub, c_sub = divmod(sub, 3)
            for r_cell in range(3):
                for c_cell in range(3):
                    grid[r_sub*3 + r_cell, c_sub*3 + c_cell] = 1.0
    return grid


def legal_to_9x9(legal_mask):
    """Convertit le masque légal 81 en grille 9×9."""
    grid = np.zeros((9, 9), dtype=np.float32)
    for sub in range(9):
        r_sub, c_sub = divmod(sub, 3)
        for cell in range(9):
            r_cell, c_cell = divmod(cell, 3)
            idx = sub * 9 + cell
            row = r_sub * 3 + r_cell
            col = c_sub * 3 + c_cell
            grid[row, col] = legal_mask[idx]
    return grid


def active_to_9x9(active_idx):
    """Plan active_board : 1 pour la sous-grille active (ou tout si -1)."""
    grid = np.zeros((9, 9), dtype=np.float32)
    if active_idx == -1:
        grid[:] = 1.0
    else:
        r_sub, c_sub = divmod(active_idx, 3)
        grid[r_sub*3:(r_sub+1)*3, c_sub*3:(c_sub+1)*3] = 1.0
    return grid


def state_to_tensor(s):
    """Encode l'état en tenseur (8, 9, 9)."""
    board, meta_board, active_idx, current_player = decode_state_string(s)
    legal_mask = compute_legal_mask(board, meta_board, active_idx)
    
    tensor = np.zeros((8, 9, 9), dtype=np.float32)
    tensor[0] = board_to_9x9(board, 1)           # P1 pieces
    tensor[1] = board_to_9x9(board, 2)           # P2 pieces
    tensor[2] = meta_to_9x9(meta_board, 1)       # Meta P1
    tensor[3] = meta_to_9x9(meta_board, 2)       # Meta P2
    tensor[4] = legal_to_9x9(legal_mask)         # Legal moves
    tensor[5] = active_to_9x9(active_idx)        # Active board
    tensor[6] = float(current_player == 1)        # Turn (constant)
    tensor[7] = 1.0                               # Bias
    return tensor


# ---- Test de validation de l'encoding ----
def validate_encoding():
    """Teste quelques propriétés de base de l'encoding."""
    # Position initiale
    s_init = '0' * 81 + '000000000' + '9' + '1' + '0'
    t = state_to_tensor(s_init)
    assert t.shape == (8, 9, 9), f"Shape incorrect: {t.shape}"
    assert t[0].sum() == 0, "P1 pieces non vide sur position initiale"
    assert t[1].sum() == 0, "P2 pieces non vide sur position initiale"
    assert t[4].sum() == 81, "Toutes les cases doivent être légales en début de partie"
    assert t[5].sum() == 81, "Toutes les sous-grilles actives en début de partie"
    assert t[6].mean() == 1.0, "Turn doit être P1 en début de partie"
    print("✓ Encoding validé")

validate_encoding()


def compute_value_target(row):
    w, l, v = row['num_wins'], row['num_losses'], row['num_visits']
    if v == 0:
        return 0.0
    return float(w - l) / float(v)


def compute_policy_target(actions, policy_size=81, label_smoothing=0.0):
    """
    Compute policy target from MCTS actions.
    BUG FIX: exp_vals.sum() avec parenthèses.
    Utilise les visites (num_wins + num_losses + num_draws) comme proxy du MCTS visit count.
    """
    pi = np.zeros(policy_size, dtype=np.float32)
    if not actions:
        return pi
    
    visits = np.zeros(policy_size, dtype=np.float32)
    for act in actions:
        idx = act['index']
        if 0 <= idx < policy_size:
            v_a = act['num_wins'] + act['num_losses'] + act['num_draws']
            visits[idx] = float(v_a)
    
    total = visits.sum()
    if total == 0:
        return pi
    
    # Distribution proportionnelle aux visites (plus stable que Q-values)
    pi = visits / total
    
    # Label smoothing optionnel
    if label_smoothing > 0:
        n_legal = (pi > 0).sum()
        if n_legal > 0:
            pi = (1 - label_smoothing) * pi + label_smoothing / n_legal * (pi > 0).astype(np.float32)
    
    return pi


print("Fonctions d'encoding définies et validées.")

✓ Encoding validé
Fonctions d'encoding définies et validées.


In [4]:
# ============================================================
# 3. SYMÉTRIES DU PLATEAU
# ============================================================
# TTTU a 8 symétries (groupe diédral D4) :
# - 4 rotations : 0°, 90°, 180°, 270°
# - 4 réflexions : horizontal, vertical, diagonale, anti-diagonale
#
# ATTENTION : pour être correctes, les symétries doivent transformer :
# 1. Le plateau 9×9 (positions des pièces)
# 2. La policy 81-dim (probabilités sur les cases)
# 3. Le active_board (index de la sous-grille active)
# La value z n'est pas affectée par les symétries.

# Mapping de l'index (sub, cell) → position globale 9×9
def idx_to_pos(idx):
    """Convertit index 0-80 en (row, col) dans la grille 9×9."""
    sub = idx // 9
    cell = idx % 9
    r_sub, c_sub = divmod(sub, 3)
    r_cell, c_cell = divmod(cell, 3)
    return r_sub * 3 + r_cell, c_sub * 3 + c_cell

def pos_to_idx(row, col):
    """Convertit (row, col) en index 0-80."""
    r_sub, r_cell = divmod(row, 3)
    c_sub, c_cell = divmod(col, 3)
    sub = r_sub * 3 + c_sub
    cell = r_cell * 3 + c_cell
    return sub * 9 + cell

# Précomputer le mapping de permutation pour chaque transformation
def build_permutation(transform_fn):
    """Construit le vecteur de permutation 81→81 pour une transformation 9×9."""
    perm = np.zeros(81, dtype=np.int64)
    for idx in range(81):
        row, col = idx_to_pos(idx)
        new_row, new_col = transform_fn(row, col)
        perm[idx] = pos_to_idx(new_row, new_col)
    return perm

# Les 8 transformations (sur grille 9×9)
transforms = [
    lambda r, c: (r, c),               # identité
    lambda r, c: (c, 8 - r),           # rotation 90° CW
    lambda r, c: (8 - r, 8 - c),       # rotation 180°
    lambda r, c: (8 - c, r),           # rotation 270° CW
    lambda r, c: (r, 8 - c),           # réflexion verticale
    lambda r, c: (8 - r, c),           # réflexion horizontale
    lambda r, c: (c, r),               # transposition diagonale
    lambda r, c: (8 - c, 8 - r),       # transposition anti-diagonale
]

PERMUTATIONS = [build_permutation(t) for t in transforms]

# Également, le sub-board index change selon la transformation
def build_sub_permutation(transform_fn):
    """Permutation des 9 sous-grilles."""
    perm = np.zeros(9, dtype=np.int64)
    for sub in range(9):
        r_sub, c_sub = divmod(sub, 3)
        # Centre de la sous-grille
        r_center, c_center = r_sub * 3 + 1, c_sub * 3 + 1
        new_r, new_c = transform_fn(r_center, c_center)
        new_r_sub, new_c_sub = new_r // 3, new_c // 3
        perm[sub] = new_r_sub * 3 + new_c_sub
    return perm

SUB_PERMUTATIONS = [build_sub_permutation(t) for t in transforms]

# Validation des permutations
def validate_permutations():
    for k, perm in enumerate(PERMUTATIONS):
        assert len(perm) == 81, f"Permutation {k}: longueur incorrecte"
        assert len(np.unique(perm)) == 81, f"Permutation {k}: pas une bijection!"
        # L'identité doit être perm[idx]=idx
        if k == 0:
            assert (perm == np.arange(81)).all(), "L'identité n'est pas correcte"
        # Aller-retour : appliquer 2 fois la rotation 180°
        if k == 2:  # rotation 180°
            double = perm[perm]
            assert (double == np.arange(81)).all(), "Rotation 180° n'est pas involutive"
    print(f"✓ {len(PERMUTATIONS)} permutations validées (bijections correctes)")

validate_permutations()


def apply_symmetry(tensor, policy, active_idx, transform_idx):
    """
    Applique une symétrie au triplet (tensor, policy, active_idx).
    tensor: (8, 9, 9)
    policy: (81,)
    active_idx: int (-1 ou 0-8)
    """
    perm = PERMUTATIONS[transform_idx]      # idx original -> idx transformé
    sub_perm = SUB_PERMUTATIONS[transform_idx]
    t = transforms[transform_idx]
    
    # Transformer le tenseur 9×9
    new_tensor = np.zeros_like(tensor)
    for r in range(9):
        for c in range(9):
            new_r, new_c = t(r, c)
            new_tensor[:, new_r, new_c] = tensor[:, r, c]
    
    # Transformer la policy
    new_policy = np.zeros_like(policy)
    new_policy[perm] = policy          # new_policy[destination] = policy[source]
    
    # Transformer l'active_idx
    if active_idx == -1:
        new_active = -1
    else:
        new_active = sub_perm[active_idx]
    
    return new_tensor, new_policy, new_active


# Test de la symétrie
def validate_symmetry_robust():    
    # Test 1 : bijection de la policy sous rotation 90°
    pi = np.zeros(81, dtype=np.float32)
    pi[0] = 1.0  # case (0,0)
    t_dummy = np.zeros((8, 9, 9), dtype=np.float32)
    _, pi_rot, _ = apply_symmetry(t_dummy, pi, -1, 1)  # rot 90° CW
    expected_idx = pos_to_idx(*transforms[1](0, 0))  # (0,0) -> (0,8)
    assert pi_rot[expected_idx] == 1.0, f"Policy rot90: attendu idx={expected_idx}, trouvé {pi_rot.argmax()}"
    assert pi_rot.sum() == pytest.approx(1.0), "Policy doit sommer à 1"
    
    # Test 2 : double rotation 180° = identité sur la policy
    pi2 = np.random.rand(81).astype(np.float32); pi2 /= pi2.sum()
    _, pi_180, _ = apply_symmetry(t_dummy, pi2, -1, 2)
    _, pi_360, _ = apply_symmetry(t_dummy, pi_180, -1, 2)
    np.testing.assert_allclose(pi_360, pi2, atol=1e-6, err_msg="Rot180° x2 ≠ identité")
    
    # Test 3 : active_idx correctement transformé
    t_active = np.zeros((8, 9, 9), dtype=np.float32)
    t_active[5] = active_to_9x9(0)  # sous-grille 0 (haut-gauche)
    t_rot, _, new_act = apply_symmetry(t_active, np.zeros(81), 0, 2)  # rot 180°
    assert new_act == 8, f"active_idx rot180°: attendu 8, trouvé {new_act}"
    assert t_rot[5, 6, 6] == 1.0, "Plan 5 mal transformé"
    
    # Test 4 : toutes les 8 symétries sont des bijections de la policy
    pi3 = np.random.rand(81).astype(np.float32); pi3 /= pi3.sum()
    for k in range(8):
        _, pi_k, _ = apply_symmetry(t_dummy, pi3, -1, k)
        np.testing.assert_allclose(pi_k.sum(), 1.0, atol=1e-6, err_msg=f"Sym {k}: policy ne somme pas à 1")
        assert len(np.unique(pi_k.nonzero())) == len(np.unique(pi3.nonzero())), f"Sym {k}: support policy changé"
    
    print("✅ Toutes les symétries validées (policy + active_idx + tenseur)")

validate_symmetry_robust()

✓ 8 permutations validées (bijections correctes)
✅ Toutes les symétries validées (policy + active_idx + tenseur)


In [5]:
# ============================================================
# 4. DATASET PYTORCH
# ============================================================
print("Chargement du dataset u3t (markstanl/u3t)...")
raw = load_dataset("markstanl/u3t")
print(raw)


class UTTTDataset(Dataset):
    """
    Dataset PyTorch amélioré pour UTTT.
    - Utilise toutes les données disponibles
    - Augmentation par symétries (8x)
    - Policy target basée sur les visites (plus stable)
    - Bug de normalisation corrigé
    """
    def __init__(self, hf_split, min_visits=50, max_samples=None, seed=42,
                 use_symmetry=False, label_smoothing=0.0):
        print(f"  Filtrage (min_visits={min_visits})...")
        filtered = hf_split.filter(
            lambda x: x["num_visits"] >= min_visits,
            num_proc=4,
            desc="Filtrage"
        )
        print(f"  Après filtrage : {len(filtered):,} exemples")
        
        if max_samples and len(filtered) > max_samples:
            rng = np.random.default_rng(seed)
            idx = rng.choice(len(filtered), size=max_samples, replace=False)
            filtered = filtered.select(idx)
            print(f"  Sous-échantillonné : {len(filtered):,} exemples")
        
        self.data = filtered
        self.use_symmetry = use_symmetry
        self.label_smoothing = label_smoothing
        # Si symétrie, chaque exemple a 8 versions (1 originale + 7 transformées)
        self.n_base = len(self.data)
        self.n_sym = 8 if use_symmetry else 1
        print(f"  Dataset final : {self.n_base:,} exemples × {self.n_sym} symétries = {len(self):,} total")

    def __len__(self):
        return self.n_base * self.n_sym

    def __getitem__(self, idx):
        base_idx = idx // self.n_sym
        sym_idx = idx % self.n_sym
        
        row = self.data[base_idx]
        
        # Value target (invariant aux symétries)
        z = compute_value_target(row)
        
        # Policy target
        pi = compute_policy_target(
            row["actions"],
            policy_size=81,
            label_smoothing=self.label_smoothing
        )
        
        # State tensor
        t = state_to_tensor(row["state"])
        
        # Active index (nécessaire pour la symétrie)
        _, _, active_idx, _ = decode_state_string(row["state"])
        
        # Appliquer la symétrie si demandé
        if self.use_symmetry and sym_idx > 0:
            t, pi, active_idx = apply_symmetry(t, pi, active_idx, sym_idx)
        
        x = torch.from_numpy(t)                                    # (8, 9, 9)
        z = torch.tensor(z, dtype=torch.float32)                   # scalar
        pi = torch.from_numpy(pi)                                  # (81,)
        
        return x, z, pi


print("\nCréation des datasets...")
print("\n[TRAIN]")
train_ds = UTTTDataset(
    raw["train"],
    min_visits=CFG["min_visits"],
    max_samples=CFG["max_train_samples"],
    seed=CFG["seed"],
    use_symmetry=CFG["use_symmetry"],
    label_smoothing=CFG["label_smoothing"]
)
print("\n[VALIDATION]")
val_ds = UTTTDataset(
    raw["validation"],
    min_visits=CFG["min_visits"],
    max_samples=None,
    seed=CFG["seed"],
    use_symmetry=False,  # Pas d'augmentation en val/test
    label_smoothing=0.0
)
print("\n[TEST]")
test_ds = UTTTDataset(
    raw["test"],
    min_visits=CFG["min_visits"],
    max_samples=None,
    seed=CFG["seed"],
    use_symmetry=False,
    label_smoothing=0.0
)

NUM_WORKERS = 10
train_loader = DataLoader(
    train_ds, batch_size=CFG["batch_size"],
    shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True
)
val_loader = DataLoader(
    val_ds, batch_size=CFG["batch_size"]*2,
    shuffle=False, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True
)
test_loader = DataLoader(
    test_ds, batch_size=CFG["batch_size"]*2,
    shuffle=False, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True
)
print(f"\nBatches train : {len(train_loader):,}")
print(f"Batches val   : {len(val_loader):,}")
print(f"Batches test  : {len(test_loader):,}")

Chargement du dataset u3t (markstanl/u3t)...


DatasetDict({
    train: Dataset({
        features: ['state', 'num_visits', 'num_wins', 'num_draws', 'num_losses', 'actions', 'depth'],
        num_rows: 5601458
    })
    test: Dataset({
        features: ['state', 'num_visits', 'num_wins', 'num_draws', 'num_losses', 'actions', 'depth'],
        num_rows: 1600367
    })
    validation: Dataset({
        features: ['state', 'num_visits', 'num_wins', 'num_draws', 'num_losses', 'actions', 'depth'],
        num_rows: 800165
    })
})

Création des datasets...

[TRAIN]
  Filtrage (min_visits=999999)...
  Après filtrage : 5,601,458 exemples
  Sous-échantillonné : 2,500,000 exemples
  Dataset final : 2,500,000 exemples × 8 symétries = 20,000,000 total

[VALIDATION]
  Filtrage (min_visits=999999)...
  Après filtrage : 800,165 exemples
  Dataset final : 800,165 exemples × 1 symétries = 800,165 total

[TEST]
  Filtrage (min_visits=999999)...
  Après filtrage : 1,600,367 exemples
  Dataset final : 1,600,367 exemples × 1 symétries = 1,600,367 t

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 10 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


In [6]:
# ============================================================
# 5. ARCHITECTURE — RESNET RÉDUIT AVEC SE BLOCKS
# ============================================================

class SEBlock(nn.Module):
    """Squeeze-Excitation block pour re-calibrer les feature maps."""
    def __init__(self, channels, reduction=8):
        super().__init__()
        self.squeeze = nn.AdaptiveAvgPool2d(1)
        self.excite = nn.Sequential(
            nn.Flatten(),
            nn.Linear(channels, max(channels // reduction, 8)),
            nn.ReLU(inplace=True),
            nn.Linear(max(channels // reduction, 8), channels),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        s = self.squeeze(x)           # (B, C, 1, 1)
        e = self.excite(s)            # (B, C)
        return x * e.view(-1, x.size(1), 1, 1)


class ResidualBlock(nn.Module):
    """Bloc résiduel avec BN, ReLU et SE optionnel."""
    def __init__(self, num_filters, use_se=True):
        super().__init__()
        self.conv1 = nn.Conv2d(num_filters, num_filters, 3, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(num_filters)
        self.conv2 = nn.Conv2d(num_filters, num_filters, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(num_filters)
        self.se    = SEBlock(num_filters) if use_se else nn.Identity()

    def forward(self, x):
        residual = x
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        x = self.se(x)
        return F.relu(x + residual)


class UTTTNetV3(nn.Module):
    """
    Réseau dual-head amélioré pour Ultimate Tic-Tac-Toe.
    
    Input  : (B, 8, 9, 9)
    Output : value (B,) ∈ [-1,1]  +  policy (B, 81) (log-probs)
    
    Améliorations vs v2:
    - 6 blocs × 128 filtres (vs 10 × 256) → ~6x moins de params
    - SE blocks pour la calibration des features
    - Dropout dans les têtes pour régulariser
    """
    def __init__(self, in_channels=8, num_filters=128, num_res_blocks=6,
                 policy_size=81, use_se=True, dropout=0.1):
        super().__init__()
        
        # Stem
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, num_filters, 3, padding=1, bias=False),
            nn.BatchNorm2d(num_filters),
            nn.ReLU(inplace=True)
        )
        
        # Residual blocks
        self.res_blocks = nn.Sequential(
            *[ResidualBlock(num_filters, use_se=use_se) for _ in range(num_res_blocks)]
        )
        
        # Policy head
        self.policy_conv = nn.Conv2d(num_filters, 4, 1, bias=False)  # 4 filtres (vs 2)
        self.policy_bn   = nn.BatchNorm2d(4)
        self.policy_drop = nn.Dropout(dropout)
        self.policy_fc   = nn.Linear(4 * 9 * 9, policy_size)
        
        # Value head
        self.value_conv  = nn.Conv2d(num_filters, 2, 1, bias=False)  # 2 filtres (vs 1)
        self.value_bn    = nn.BatchNorm2d(2)
        self.value_drop  = nn.Dropout(dropout)
        self.value_fc1   = nn.Linear(2 * 9 * 9, 128)  # réduit vs 256
        self.value_fc2   = nn.Linear(128, 1)
        
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = self.stem(x)
        x = self.res_blocks(x)
        
        # Policy head
        p = F.relu(self.policy_bn(self.policy_conv(x)))
        p = self.policy_drop(p.view(p.size(0), -1))
        p = F.log_softmax(self.policy_fc(p), dim=1)
        
        # Value head
        v = F.relu(self.value_bn(self.value_conv(x)))
        v = self.value_drop(v.view(v.size(0), -1))
        v = F.relu(self.value_fc1(v))
        v = torch.tanh(self.value_fc2(v)).squeeze(1)
        
        return p, v


model = UTTTNetV3(
    in_channels   = CFG["in_channels"],
    num_filters   = CFG["num_filters"],
    num_res_blocks= CFG["num_res_blocks"],
    policy_size   = CFG["policy_size"],
    use_se        = CFG["use_se"],
    dropout       = CFG["dropout"]
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
print(f"Paramètres total : {n_params:,}  (~{n_params/1e6:.1f}M)")

# Test forward
with torch.no_grad():
    test_x = torch.randn(4, CFG["in_channels"], 9, 9).to(DEVICE)
    test_p, test_v = model(test_x)
print(f"Policy shape : {test_p.shape}  (log-probs)")
print(f"Value  shape : {test_v.shape}")
assert test_p.shape == (4, 81)
assert test_v.shape == (4,)
print("✓ Forward pass OK")

Paramètres total : 1,855,554  (~1.9M)
Policy shape : torch.Size([4, 81])  (log-probs)
Value  shape : torch.Size([4])
✓ Forward pass OK


In [7]:
# ============================================================
# 6. LOSS, OPTIMISEUR & SCHEDULER AVEC WARM-UP
# ============================================================

def compute_loss(policy_logprob, value_pred, z_target, pi_target, lambda_policy=1.0):
    """
    L = MSE(value, z) + λ * KL(pi_target || policy)
    """
    # Value loss (MSE)
    value_loss = F.mse_loss(value_pred, z_target)
    
    # Policy loss (KL divergence) — seulement sur les états avec coups légaux
    has_legal = (pi_target.sum(dim=1) > 0)
    if has_legal.sum() > 0:
        policy_loss = F.kl_div(
            policy_logprob[has_legal],
            pi_target[has_legal],
            reduction='batchmean',
            log_target=False
        )
    else:
        policy_loss = torch.tensor(0.0, device=value_pred.device)
    
    total_loss = value_loss + lambda_policy * policy_loss
    return total_loss, value_loss.item(), policy_loss.item()


optimizer = AdamW(
    model.parameters(),
    lr=CFG["lr"],
    weight_decay=CFG["weight_decay"]
)

# Scheduler avec warm-up linéaire puis cosine decay
n_epochs = CFG["epochs"]
n_warmup = CFG["warmup_epochs"]

def warmup_lambda(epoch):
    if epoch < n_warmup:
        return (epoch + 1) / n_warmup
    return 1.0

warmup_scheduler = LambdaLR(optimizer, lr_lambda=warmup_lambda)
cosine_scheduler = CosineAnnealingLR(
    optimizer,
    T_max=n_epochs - n_warmup,
    eta_min=1e-6
)
scheduler = SequentialLR(
    optimizer,
    schedulers=[warmup_scheduler, cosine_scheduler],
    milestones=[n_warmup]
)

# Mixed precision scaler
scaler = GradScaler() if DEVICE.type == "cuda" else None

# ---- Reprise depuis checkpoint ----
CHECKPOINT_PATH      = "best_uttt_v3_model.pth"
RESUME_CHECKPOINT    = "resume_uttt_v3.pth"   # checkpoint complet pour reprise

history = {
    "train_total": [], "train_value": [], "train_policy": [],
    "val_total":   [], "val_value":   [], "val_policy":   [],
    "train_pearson": [], "val_pearson": [],
    "train_dir_acc": [], "val_dir_acc": [],
    "lr":          []
}
best_val_loss = float("inf")
start_epoch   = 1

if os.path.exists(RESUME_CHECKPOINT):
    print(f"Reprise depuis '{RESUME_CHECKPOINT}'...")
    ckpt = torch.load(RESUME_CHECKPOINT, map_location=DEVICE, weights_only = False)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    scheduler.load_state_dict(ckpt["scheduler"])
    if scaler is not None and ckpt.get("scaler") is not None:
        scaler.load_state_dict(ckpt["scaler"])
    history       = ckpt["history"]
    best_val_loss = ckpt["best_val_loss"]
    start_epoch   = ckpt["epoch"] + 1
    print(f"  Reprise à l'epoch {start_epoch}  (best val loss = {best_val_loss:.4f})")
else:
    print("Aucun checkpoint de reprise trouvé — entraînement from scratch.")

print(f"Optimiseur : AdamW  lr={CFG['lr']}  wd={CFG['weight_decay']}")
print(f"Scheduler  : {n_warmup} epochs warmup + CosineAnnealing({n_epochs - n_warmup} epochs)")
print(f"Mixed precision : {'OUI' if scaler else 'NON'} (GradScaler)")

Reprise depuis 'resume_uttt_v3.pth'...
  Reprise à l'epoch 8  (best val loss = 1.0160)
Optimiseur : AdamW  lr=0.0003  wd=0.0001
Scheduler  : 3 epochs warmup + CosineAnnealing(17 epochs)
Mixed precision : OUI (GradScaler)


/tmp/ipykernel_793/2742579390.py:56: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler() if DEVICE.type == "cuda" else None


In [8]:
print(history)

{'train_total': [2.381645806316405, 1.6816578948908165, 1.2981535810641742, 1.1304106876383286, 1.0542210675098187, 1.0096557085684514, 0.9777167443345185], 'train_value': [0.09610511088057568, 0.08992783209687004, 0.08726328784859672, 0.08304029991825462, 0.07777205484642886, 0.07318421315989997, 0.06970209802287099], 'train_policy': [2.2855406954358317, 1.5917300627939484, 1.210890293215577, 1.0473703877200755, 0.9764490126633917, 0.9364714954085508, 0.908014646311648], 'val_total': [1.9610507305787535, 1.5608016544458818, 1.2613441019642109, 1.2157528534227489, 1.4142358692324892, 1.1999098281471097, 1.0160476705249475], 'val_value': [0.09140169339216485, 0.0902249058138351, 0.08581072761088002, 0.08813096674121156, 0.07828558243963184, 0.0759615967316287, 0.06837784111195681], 'val_policy': [1.8696490331571929, 1.470576750988863, 1.175533376177963, 1.1276218830322733, 1.3359502845880937, 1.1239482286025066, 0.947669832682123], 'train_pearson': [np.float32(0.4204222), np.float32(0.4

In [ ]:
# ============================================================
# 7. BOUCLE D'ENTRAÎNEMENT AVEC MÉTRIQUES ÉTENDUES
# ============================================================

def run_epoch(loader, model, optimizer=None, lambda_p=1.0, desc="", scaler=None):
    """Exécute une epoch avec mixed precision optionnel."""
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    
    total_loss = val_loss = pol_loss = 0.0
    all_values_pred = []
    all_values_true = []
    n_batches = 0
    
    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for x, z, pi in tqdm(loader, desc=desc, leave=False):
            x  = x.to(DEVICE, non_blocking=True)
            z  = z.to(DEVICE, non_blocking=True)
            pi = pi.to(DEVICE, non_blocking=True)
            
            if is_train and scaler is not None:
                with autocast():
                    policy_lp, value = model(x)
                    loss, v_l, p_l = compute_loss(policy_lp, value, z, pi, lambda_p)
                optimizer.zero_grad()
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                policy_lp, value = model(x)
                loss, v_l, p_l = compute_loss(policy_lp, value, z, pi, lambda_p)
                if is_train:
                    optimizer.zero_grad()
                    loss.backward()
                    nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    optimizer.step()
            
            total_loss += loss.item()
            val_loss   += v_l
            pol_loss   += p_l
            n_batches  += 1
            
            all_values_pred.extend(value.detach().cpu().numpy())
            all_values_true.extend(z.cpu().numpy())
    
    # Métriques sur la value head
    preds = np.array(all_values_pred)
    trues = np.array(all_values_true)
    pearson_r = scipy.stats.pearsonr(trues, preds)[0] if len(preds) > 1 else 0.0
    dir_acc = np.mean((preds * trues) > 0) if len(preds) > 0 else 0.0
    
    return (
        total_loss / n_batches,
        val_loss   / n_batches,
        pol_loss   / n_batches,
        pearson_r,
        dir_acc
    )


print("\n" + "="*75)
print(" ENTRAÎNEMENT")
print("="*75)
print(f"{'Epoch':>5} | {'Train Loss':>10} | {'Val Loss':>9} | {'Val Pearson':>11} | {'Val Dir%':>8} | {'LR':>8}")
print("-"*75)

# Réafficher l'historique déjà effectué si reprise
for i, ep in enumerate(range(1, start_epoch)):
    print(f"{ep:>5} | {history['train_total'][i]:>10.4f} | {history['val_total'][i]:>9.4f} | "
          f"{history['val_pearson'][i]:>11.4f} | {history['val_dir_acc'][i]*100:>7.1f}% | "
          f"{history['lr'][i]:>8.2e}  (déjà effectuée)")

for epoch in range(start_epoch, n_epochs + 1):
    tr_total, tr_val, tr_pol, tr_r, tr_da = run_epoch(
        train_loader, model, optimizer,
        lambda_p=CFG["lambda_policy"],
        desc=f"Ep {epoch:02d}/train",
        scaler=scaler
    )
    va_total, va_val, va_pol, va_r, va_da = run_epoch(
        val_loader, model, optimizer=None,
        lambda_p=CFG["lambda_policy"],
        desc=f"Ep {epoch:02d}/val  "
    )
    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']
    
    if va_total < best_val_loss:
        best_val_loss = va_total
        torch.save(model.state_dict(), CHECKPOINT_PATH)
        improved = " ✓"
    else:
        improved = ""
    
    history["train_total"].append(tr_total)
    history["train_value"].append(tr_val)
    history["train_policy"].append(tr_pol)
    history["val_total"].append(va_total)
    history["val_value"].append(va_val)
    history["val_policy"].append(va_pol)
    history["train_pearson"].append(tr_r)
    history["val_pearson"].append(va_r)
    history["train_dir_acc"].append(tr_da)
    history["val_dir_acc"].append(va_da)
    history["lr"].append(current_lr)
    
    print(f"{epoch:>5} | {tr_total:>10.4f} | {va_total:>9.4f} | {va_r:>11.4f} | {va_da*100:>7.1f}% | {current_lr:>8.2e}{improved}")
    
    # Sauvegarde complète pour reprise (écrase à chaque epoch)
    resume_ckpt = {
        "epoch":         epoch,
        "model":         model.state_dict(),
        "optimizer":     optimizer.state_dict(),
        "scheduler":     scheduler.state_dict(),
        "scaler":        scaler.state_dict() if scaler is not None else None,
        "history":       history,
        "best_val_loss": best_val_loss,
    }
    torch.save(resume_ckpt, RESUME_CHECKPOINT)

print(f"\nMeilleure val loss : {best_val_loss:.4f}")


 ENTRAÎNEMENT
Epoch | Train Loss |  Val Loss | Val Pearson | Val Dir% |       LR
---------------------------------------------------------------------------
    1 |     2.3816 |    1.9611 |      0.4714 |    72.8% | 2.00e-04  (déjà effectuée)
    2 |     1.6817 |    1.5608 |      0.4838 |    72.8% | 3.00e-04  (déjà effectuée)
    3 |     1.2982 |    1.2613 |      0.5178 |    73.0% | 3.00e-04  (déjà effectuée)
    4 |     1.1304 |    1.2158 |      0.5269 |    73.7% | 2.97e-04  (déjà effectuée)
    5 |     1.0542 |    1.4142 |      0.5785 |    73.8% | 2.90e-04  (déjà effectuée)
    6 |     1.0097 |    1.1999 |      0.5942 |    74.6% | 2.78e-04  (déjà effectuée)
    7 |     0.9777 |    1.0160 |      0.6458 |    75.9% | 2.61e-04  (déjà effectuée)


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 10 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Ep 08/train:   0%|          | 0/4883 [00:02<?, ?it/s]

/tmp/ipykernel_793/3066175837.py:23: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Ep 08/val  :   0%|          | 0/98 [00:00<?, ?it/s]

...

   13 |     0.8697 |    0.9543 |      0.6920 |    76.5% | 1.10e-04 ✓


Ep 14/train:   0%|          | 0/4883 [00:02<?, ?it/s]

Ep 14/val  :   0%|          | 0/98 [00:00<?, ?it/s]

   14 |     0.8589 |    0.9116 |      0.7018 |    77.2% | 8.39e-05 ✓


Ep 15/train:   0%|          | 0/4883 [00:02<?, ?it/s]

In [ ]:
# ============================================================
# 8. ÉVALUATION FINALE SUR LE TEST SET
# ============================================================
model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE))
print(f"Meilleur modèle chargé depuis '{CHECKPOINT_PATH}'")

te_total, te_val, te_pol, te_r, te_da = run_epoch(
    test_loader, model, optimizer=None,
    lambda_p=CFG["lambda_policy"],
    desc="Test set"
)

print("\n" + "="*65)
print(" RÉSULTATS FINAUX")
print("="*65)
print(f"{'Split':<10} {'Total Loss':>10} {'Value MSE':>10} {'Policy KL':>10} {'Pearson r':>10} {'Dir Acc':>8}")
print("-"*65)
print(f"{'Train':<10} {history['train_total'][-1]:>10.4f} {history['train_value'][-1]:>10.4f} {history['train_policy'][-1]:>10.4f} {history['train_pearson'][-1]:>10.4f} {history['train_dir_acc'][-1]*100:>7.1f}%")
print(f"{'Val':<10} {history['val_total'][-1]:>10.4f} {history['val_value'][-1]:>10.4f} {history['val_policy'][-1]:>10.4f} {history['val_pearson'][-1]:>10.4f} {history['val_dir_acc'][-1]*100:>7.1f}%")
print(f"{'Test':<10} {te_total:>10.4f} {te_val:>10.4f} {te_pol:>10.4f} {te_r:>10.4f} {te_da*100:>7.1f}%")
print("-"*65)
print(f"\nBest val loss    : {best_val_loss:.4f}")
print(f"Paramètres       : {sum(p.numel() for p in model.parameters()):,}")
print(f"Architecture     : {CFG['num_res_blocks']} blocs × {CFG['num_filters']} filtres + SE")
print(f"Symétries        : {'OUI (×8)' if CFG['use_symmetry'] else 'NON'}")

In [ ]:
# ============================================================
# 9. VISUALISATION
# ============================================================
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Courbes d'entraînement — UTTT ResNet v3", fontsize=14, fontweight='bold')

ep = range(1, len(history['train_total']) + 1)

# Loss totale
ax = axes[0, 0]
ax.plot(ep, history['train_total'], 'b-o', label='Train', ms=4)
ax.plot(ep, history['val_total'],   'r-o', label='Val',   ms=4)
ax.axhline(te_total, color='g', ls='--', label=f'Test={te_total:.4f}')
ax.set_title('Loss Totale'); ax.set_xlabel('Epoch'); ax.legend(); ax.grid(alpha=0.3)

# Value MSE
ax = axes[0, 1]
ax.plot(ep, history['train_value'], 'b-o', ms=4)
ax.plot(ep, history['val_value'],   'r-o', ms=4)
ax.axhline(te_val, color='g', ls='--', label=f'Test={te_val:.4f}')
ax.set_title('Value Loss (MSE)'); ax.set_xlabel('Epoch'); ax.legend(); ax.grid(alpha=0.3)

# Policy KL
ax = axes[0, 2]
ax.plot(ep, history['train_policy'], 'b-o', ms=4)
ax.plot(ep, history['val_policy'],   'r-o', ms=4)
ax.axhline(te_pol, color='g', ls='--', label=f'Test={te_pol:.4f}')
ax.set_title('Policy Loss (KL)'); ax.set_xlabel('Epoch'); ax.legend(); ax.grid(alpha=0.3)

# Pearson correlation
ax = axes[1, 0]
ax.plot(ep, history['train_pearson'], 'b-o', ms=4, label='Train')
ax.plot(ep, history['val_pearson'],   'r-o', ms=4, label='Val')
ax.axhline(te_r, color='g', ls='--', label=f'Test={te_r:.4f}')
ax.set_title('Pearson Correlation (Value)'); ax.set_xlabel('Epoch'); ax.legend(); ax.grid(alpha=0.3)

# Directional accuracy
ax = axes[1, 1]
ax.plot(ep, [x*100 for x in history['train_dir_acc']], 'b-o', ms=4, label='Train')
ax.plot(ep, [x*100 for x in history['val_dir_acc']],   'r-o', ms=4, label='Val')
ax.axhline(te_da*100, color='g', ls='--', label=f'Test={te_da*100:.1f}%')
ax.set_title('Directional Accuracy (%)'); ax.set_xlabel('Epoch'); ax.legend(); ax.grid(alpha=0.3)

# Learning rate
ax = axes[1, 2]
ax.plot(ep, history['lr'], 'purple', lw=2)
ax.set_title('Learning Rate'); ax.set_xlabel('Epoch'); ax.set_yscale('log'); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves_v3.png', dpi=150, bbox_inches='tight')
plt.show()
print("Courbes sauvegardées : training_curves_v3.png")

In [ ]:
# ============================================================
# 10. EXPORT DU MODÈLE POUR LE MOTEUR DE JEU
# ============================================================
# Exporter en TorchScript pour l'intégration C++/Python

model.eval()
dummy_input = torch.randn(1, CFG["in_channels"], 9, 9).to(DEVICE)

try:
    scripted = torch.jit.trace(model, dummy_input)
    torch.jit.save(scripted, "uttt_v3_model_scripted.pt")
    print("Modèle TorchScript sauvegardé : uttt_v3_model_scripted.pt")
    
    # Test du modèle scripté
    output_p, output_v = scripted(dummy_input)
    assert output_p.shape == (1, 81)
    assert output_v.shape == (1,)
    print("✓ Modèle scripté validé")
except Exception as e:
    print(f"TorchScript export failed: {e}")
    print("Utiliser le checkpoint PyTorch standard : " + CHECKPOINT_PATH)

# Afficher les infos pour l'intégration dans le moteur
print("\n=" * 50)
print("GUIDE D'INTÉGRATION MOTEUR")
print("=" * 50)
print(f"Input  : torch.Tensor(B, 8, 9, 9) float32")
print(f"Output : (log_probs(B,81), value(B,)) float32")
print(f"Policy : exp(log_probs) pour obtenir les probabilités")
print(f"Value  : tanh → [-1, 1], positif = avantage pour le joueur courant")
print(f"\nEncoding (8 plans) :")
print(f"  [0] Pièces joueur 1 (9×9)")
print(f"  [1] Pièces joueur 2 (9×9)")
print(f"  [2] Meta-board P1 (grandes cases gagnées)")
print(f"  [3] Meta-board P2 (grandes cases gagnées)")
print(f"  [4] Coups légaux")
print(f"  [5] Sous-grille active")
print(f"  [6] Tour (1=P1, 0=P2)")
print(f"  [7] Constante 1.0")